# Customer Success Prediction Agent

**Framework**: Scikit-learn + Pandas + OpenAI | **Models**: Machine Learning + GPT-4 Insights

An intelligent customer success prediction system that uses machine learning and AI to predict customer churn, identify expansion opportunities, and recommend proactive interventions. Combines traditional ML models with AI-powered insights for comprehensive customer success management.

## Features

- **Churn Prediction**: ML models to predict customer churn probability
- **Health Scoring**: Comprehensive customer health assessment
- **Expansion Identification**: Predict upsell and cross-sell opportunities
- **Risk Analysis**: Early warning system for at-risk accounts
- **Intervention Recommendations**: AI-powered action suggestions
- **Success Tracking**: Monitor customer journey and outcomes

## Architecture

The system combines multiple approaches:

1. **Data Processing**: Customer data aggregation and feature engineering
2. **ML Models**: Churn prediction, health scoring, opportunity detection
3. **AI Analysis**: GPT-4 powered insights and recommendations
4. **Risk Assessment**: Multi-factor risk evaluation
5. **Action Planning**: Automated intervention recommendations
6. **Outcome Tracking**: Success metric monitoring and optimization

## Installation

```bash
pip install scikit-learn pandas numpy openai anthropic matplotlib seaborn plotly xgboost
```

## Setup

Configure your environment and API keys:

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum
import uuid
import random
import warnings
from collections import defaultdict
import pickle

# Machine Learning imports
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif
import xgboost as xgb

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
warnings.filterwarnings('ignore')

# AI Model clients
from openai import OpenAI
try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False

# Initialize AI clients
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
if ANTHROPIC_AVAILABLE:
    anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("✅ Customer Success Prediction Agent initialized!")
print(f"📊 Machine Learning libraries loaded")
print(f"🤖 AI analysis capabilities ready")
print(f"🧠 Anthropic available: {ANTHROPIC_AVAILABLE}")

## Core Data Models

In [ ]:
class CustomerTier(Enum):
    ENTERPRISE = "enterprise"
    PROFESSIONAL = "professional"
    STARTER = "starter"
    TRIAL = "trial"

class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class HealthStatus(Enum):
    HEALTHY = "healthy"
    AT_RISK = "at_risk"
    UNHEALTHY = "unhealthy"
    CRITICAL = "critical"

@dataclass
class Customer:
    id: str
    name: str
    email: str
    company: str
    tier: CustomerTier
    signup_date: datetime
    mrr: float  # Monthly Recurring Revenue
    contract_value: float
    contract_end_date: datetime
    industry: str
    company_size: int
    csm_assigned: Optional[str] = None  # Customer Success Manager
    is_churned: bool = False
    churn_date: Optional[datetime] = None
    created_at: datetime = field(default_factory=datetime.now)

@dataclass
class UsageMetrics:
    customer_id: str
    date: datetime
    daily_active_users: int
    monthly_active_users: int
    feature_adoption_score: float  # 0-100
    session_duration_avg: float  # minutes
    sessions_per_day: float
    api_calls: int
    support_tickets: int
    nps_score: Optional[float] = None  # Net Promoter Score

@dataclass
class EngagementMetrics:
    customer_id: str
    date: datetime
    email_opens: int
    email_clicks: int
    webinar_attendance: int
    documentation_views: int
    community_posts: int
    training_completed: int
    csm_interactions: int
    last_login_days_ago: int

@dataclass
class CustomerHealth:
    customer_id: str
    health_score: float  # 0-100
    health_status: HealthStatus
    churn_probability: float  # 0-1
    risk_level: RiskLevel
    expansion_probability: float  # 0-1
    key_risk_factors: List[str]
    opportunities: List[str]
    recommended_actions: List[str]
    last_updated: datetime = field(default_factory=datetime.now)

@dataclass
class Prediction:
    customer_id: str
    prediction_type: str  # churn, expansion, health
    probability: float
    confidence: float
    model_used: str
    feature_importance: Dict[str, float]
    created_at: datetime = field(default_factory=datetime.now)

print("✅ Data models defined successfully!")

## Data Generation and Feature Engineering

In [ ]:
class CustomerDataGenerator:
    """Generate realistic customer data for model training and testing."""
    
    def __init__(self, num_customers: int = 1000):
        self.num_customers = num_customers
        self.customers = []
        self.usage_data = []
        self.engagement_data = []
        
        # Industry and company data
        self.industries = [
            'Technology', 'Healthcare', 'Finance', 'Manufacturing', 'Retail',
            'Education', 'Real Estate', 'Media', 'Consulting', 'Other'
        ]
        
        self.company_names = [
            'TechCorp', 'DataSystems', 'InnovateLabs', 'CloudFirst', 'DigitalWorks',
            'FutureLogic', 'SmartSolutions', 'NextGen', 'ProActive', 'DynamicTech'
        ]
    
    def generate_customers(self) -> List[Customer]:
        """Generate customer data with realistic distributions."""
        customers = []
        
        for i in range(self.num_customers):
            # Generate customer attributes
            customer_id = f"cust_{i+1:04d}"
            tier = random.choice(list(CustomerTier))
            
            # MRR based on tier
            mrr_ranges = {
                CustomerTier.ENTERPRISE: (5000, 50000),
                CustomerTier.PROFESSIONAL: (500, 5000),
                CustomerTier.STARTER: (50, 500),
                CustomerTier.TRIAL: (0, 0)
            }
            
            mrr = random.uniform(*mrr_ranges[tier])
            contract_value = mrr * random.uniform(12, 36)  # 1-3 year contracts
            
            # Signup date (last 2 years)
            signup_date = datetime.now() - timedelta(days=random.randint(30, 730))
            
            # Contract end date
            contract_length = random.choice([12, 24, 36])  # months
            contract_end_date = signup_date + timedelta(days=contract_length * 30)
            
            # Company attributes
            industry = random.choice(self.industries)
            company_size = random.choice([10, 25, 50, 100, 250, 500, 1000, 2500, 5000])
            
            # Churn simulation (realistic rates by tier)
            churn_rates = {
                CustomerTier.ENTERPRISE: 0.05,
                CustomerTier.PROFESSIONAL: 0.12,
                CustomerTier.STARTER: 0.25,
                CustomerTier.TRIAL: 0.70
            }
            
            is_churned = random.random() < churn_rates[tier]
            churn_date = None
            if is_churned:
                days_since_signup = (datetime.now() - signup_date).days
                churn_date = signup_date + timedelta(days=random.randint(30, days_since_signup))
            
            customer = Customer(
                id=customer_id,
                name=f"User {i+1}",
                email=f"user{i+1}@{random.choice(self.company_names).lower()}.com",
                company=f"{random.choice(self.company_names)} {random.choice(['Inc', 'Corp', 'LLC', 'Ltd'])}",
                tier=tier,
                signup_date=signup_date,
                mrr=mrr,
                contract_value=contract_value,
                contract_end_date=contract_end_date,
                industry=industry,
                company_size=company_size,
                csm_assigned=f"CSM_{random.randint(1, 10)}" if tier in [CustomerTier.ENTERPRISE, CustomerTier.PROFESSIONAL] else None,
                is_churned=is_churned,
                churn_date=churn_date
            )
            
            customers.append(customer)
        
        self.customers = customers
        return customers
    
    def generate_usage_metrics(self, days: int = 90) -> List[UsageMetrics]:
        """Generate usage metrics for the last N days."""
        usage_data = []
        
        for customer in self.customers:
            # Skip churned customers after churn date
            end_date = customer.churn_date if customer.is_churned else datetime.now()
            start_date = max(customer.signup_date, end_date - timedelta(days=days))
            
            current_date = start_date
            while current_date <= end_date:
                # Usage patterns based on customer tier and churn status
                base_usage = {
                    CustomerTier.ENTERPRISE: 100,
                    CustomerTier.PROFESSIONAL: 50,
                    CustomerTier.STARTER: 10,
                    CustomerTier.TRIAL: 5
                }[customer.tier]
                
                # Reduce usage for churned customers leading up to churn
                usage_multiplier = 1.0
                if customer.is_churned and customer.churn_date:
                    days_to_churn = (customer.churn_date - current_date).days
                    if days_to_churn <= 30:
                        usage_multiplier = max(0.1, 1.0 - (30 - days_to_churn) / 30 * 0.7)
                
                # Add some randomness
                usage_multiplier *= random.uniform(0.5, 1.5)
                
                daily_users = max(1, int(base_usage * usage_multiplier * random.uniform(0.8, 1.2)))
                monthly_users = max(daily_users, int(daily_users * random.uniform(3, 8)))
                
                usage = UsageMetrics(
                    customer_id=customer.id,
                    date=current_date,
                    daily_active_users=daily_users,
                    monthly_active_users=monthly_users,
                    feature_adoption_score=random.uniform(20, 95) * usage_multiplier,
                    session_duration_avg=random.uniform(5, 45) * usage_multiplier,
                    sessions_per_day=random.uniform(1, 10) * usage_multiplier,
                    api_calls=max(0, int(random.uniform(100, 10000) * usage_multiplier)),
                    support_tickets=random.poisson(0.5) if usage_multiplier > 0.5 else random.poisson(2),
                    nps_score=random.uniform(6, 10) * usage_multiplier if random.random() < 0.1 else None
                )
                
                usage_data.append(usage)
                current_date += timedelta(days=1)
        
        self.usage_data = usage_data
        return usage_data
    
    def generate_engagement_metrics(self, days: int = 90) -> List[EngagementMetrics]:
        """Generate engagement metrics."""
        engagement_data = []
        
        for customer in self.customers:
            end_date = customer.churn_date if customer.is_churned else datetime.now()
            start_date = max(customer.signup_date, end_date - timedelta(days=days))
            
            # Weekly engagement data
            current_date = start_date
            while current_date <= end_date:
                # Engagement decreases for churning customers
                engagement_multiplier = 1.0
                if customer.is_churned and customer.churn_date:
                    days_to_churn = (customer.churn_date - current_date).days
                    if days_to_churn <= 60:
                        engagement_multiplier = max(0.1, 1.0 - (60 - days_to_churn) / 60 * 0.8)
                
                engagement = EngagementMetrics(
                    customer_id=customer.id,
                    date=current_date,
                    email_opens=random.poisson(2) * engagement_multiplier,
                    email_clicks=random.poisson(1) * engagement_multiplier,
                    webinar_attendance=1 if random.random() < 0.1 * engagement_multiplier else 0,
                    documentation_views=random.poisson(3) * engagement_multiplier,
                    community_posts=1 if random.random() < 0.05 * engagement_multiplier else 0,
                    training_completed=1 if random.random() < 0.02 * engagement_multiplier else 0,
                    csm_interactions=1 if customer.csm_assigned and random.random() < 0.2 * engagement_multiplier else 0,
                    last_login_days_ago=random.randint(0, 7) if engagement_multiplier > 0.5 else random.randint(7, 30)
                )
                
                engagement_data.append(engagement)
                current_date += timedelta(days=7)  # Weekly data
        
        self.engagement_data = engagement_data
        return engagement_data

# Generate sample data
print("📊 Generating sample customer data...")
data_generator = CustomerDataGenerator(num_customers=1000)
customers = data_generator.generate_customers()
usage_metrics = data_generator.generate_usage_metrics()
engagement_metrics = data_generator.generate_engagement_metrics()

print(f"✅ Generated data for {len(customers)} customers")
print(f"📈 {len(usage_metrics)} usage data points")
print(f"👥 {len(engagement_metrics)} engagement data points")
print(f"📉 Churn rate: {sum(c.is_churned for c in customers) / len(customers):.1%}")

## Feature Engineering and Data Preparation

In [ ]:
class FeatureEngineer:
    """Engineer features for machine learning models."""
    
    def __init__(self, customers: List[Customer], usage_data: List[UsageMetrics], 
                 engagement_data: List[EngagementMetrics]):
        self.customers = customers
        self.usage_data = usage_data
        self.engagement_data = engagement_data
    
    def create_feature_dataset(self) -> pd.DataFrame:
        """Create comprehensive feature dataset for ML models."""
        features = []
        
        for customer in self.customers:
            # Customer basic features
            customer_features = {
                'customer_id': customer.id,
                'tier': customer.tier.value,
                'mrr': customer.mrr,
                'contract_value': customer.contract_value,
                'industry': customer.industry,
                'company_size': customer.company_size,
                'has_csm': 1 if customer.csm_assigned else 0,
                'is_churned': 1 if customer.is_churned else 0,
            }
            
            # Time-based features
            days_since_signup = (datetime.now() - customer.signup_date).days
            days_to_contract_end = (customer.contract_end_date - datetime.now()).days
            
            customer_features.update({
                'days_since_signup': days_since_signup,
                'days_to_contract_end': days_to_contract_end,
                'contract_length_months': days_since_signup / 30,
            })
            
            # Usage features (last 30 days)
            recent_usage = [u for u in self.usage_data 
                          if u.customer_id == customer.id and 
                          (datetime.now() - u.date).days <= 30]
            
            if recent_usage:
                customer_features.update({
                    'avg_daily_users': np.mean([u.daily_active_users for u in recent_usage]),
                    'avg_monthly_users': np.mean([u.monthly_active_users for u in recent_usage]),
                    'avg_feature_adoption': np.mean([u.feature_adoption_score for u in recent_usage]),
                    'avg_session_duration': np.mean([u.session_duration_avg for u in recent_usage]),
                    'avg_sessions_per_day': np.mean([u.sessions_per_day for u in recent_usage]),
                    'total_api_calls': sum([u.api_calls for u in recent_usage]),
                    'total_support_tickets': sum([u.support_tickets for u in recent_usage]),
                    'usage_trend': self._calculate_trend([u.daily_active_users for u in recent_usage]),
                })
                
                # NPS if available
                nps_scores = [u.nps_score for u in recent_usage if u.nps_score is not None]
                customer_features['avg_nps'] = np.mean(nps_scores) if nps_scores else 0
            else:
                # Default values for customers with no recent usage
                customer_features.update({
                    'avg_daily_users': 0, 'avg_monthly_users': 0, 'avg_feature_adoption': 0,
                    'avg_session_duration': 0, 'avg_sessions_per_day': 0, 'total_api_calls': 0,
                    'total_support_tickets': 0, 'usage_trend': 0, 'avg_nps': 0
                })
            
            # Engagement features (last 30 days)
            recent_engagement = [e for e in self.engagement_data 
                               if e.customer_id == customer.id and 
                               (datetime.now() - e.date).days <= 30]
            
            if recent_engagement:
                customer_features.update({
                    'total_email_opens': sum([e.email_opens for e in recent_engagement]),
                    'total_email_clicks': sum([e.email_clicks for e in recent_engagement]),
                    'total_webinar_attendance': sum([e.webinar_attendance for e in recent_engagement]),
                    'total_doc_views': sum([e.documentation_views for e in recent_engagement]),
                    'total_community_posts': sum([e.community_posts for e in recent_engagement]),
                    'total_training_completed': sum([e.training_completed for e in recent_engagement]),
                    'total_csm_interactions': sum([e.csm_interactions for e in recent_engagement]),
                    'avg_last_login_days': np.mean([e.last_login_days_ago for e in recent_engagement]),
                })
            else:
                customer_features.update({
                    'total_email_opens': 0, 'total_email_clicks': 0, 'total_webinar_attendance': 0,
                    'total_doc_views': 0, 'total_community_posts': 0, 'total_training_completed': 0,
                    'total_csm_interactions': 0, 'avg_last_login_days': 30
                })
            
            # Derived features
            customer_features.update({
                'email_engagement_rate': customer_features['total_email_clicks'] / max(1, customer_features['total_email_opens']),
                'support_ticket_rate': customer_features['total_support_tickets'] / max(1, customer_features['avg_daily_users']),
                'user_activity_ratio': customer_features['avg_daily_users'] / max(1, customer_features['avg_monthly_users']),
                'value_per_user': customer_features['mrr'] / max(1, customer_features['avg_monthly_users']),
            })
            
            features.append(customer_features)
        
        df = pd.DataFrame(features)
        
        # Encode categorical variables
        df = pd.get_dummies(df, columns=['tier', 'industry'], prefix=['tier', 'industry'])
        
        # Fill NaN values
        df = df.fillna(0)
        
        return df
    
    def _calculate_trend(self, values: List[float]) -> float:
        """Calculate trend slope for a series of values."""
        if len(values) < 2:
            return 0
        
        x = np.arange(len(values))
        slope = np.polyfit(x, values, 1)[0]
        return slope

# Create feature dataset
print("🔧 Engineering features for ML models...")
feature_engineer = FeatureEngineer(customers, usage_metrics, engagement_metrics)
feature_df = feature_engineer.create_feature_dataset()

print(f"✅ Created feature dataset with {len(feature_df)} rows and {len(feature_df.columns)} features")
print(f"📊 Target distribution: {feature_df['is_churned'].value_counts().to_dict()}")

# Display feature summary
print("\n📋 Feature Summary:")
print(f"  • Customer attributes: {len([c for c in feature_df.columns if 'tier_' in c or 'industry_' in c or c in ['mrr', 'contract_value', 'company_size']])}")
print(f"  • Usage features: {len([c for c in feature_df.columns if 'avg_' in c or 'total_api' in c or 'usage_' in c])}")
print(f"  • Engagement features: {len([c for c in feature_df.columns if 'email_' in c or 'webinar_' in c or 'doc_' in c or 'community_' in c])}")
print(f"  • Derived features: {len([c for c in feature_df.columns if 'rate' in c or 'ratio' in c or 'per_' in c])}")

## Machine Learning Models

In [ ]:
class ChurnPredictionModel:
    """Machine learning models for churn prediction."""
    
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.feature_selectors = {}
        self.feature_names = None
        self.is_trained = False
    
    def prepare_data(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        """Prepare data for model training."""
        # Separate features and target
        feature_cols = [col for col in df.columns if col not in ['customer_id', 'is_churned']]
        X = df[feature_cols].values
        y = df['is_churned'].values
        
        return X, y, feature_cols
    
    def train_models(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Train multiple churn prediction models."""
        print("🤖 Training churn prediction models...")
        
        X, y, feature_names = self.prepare_data(df)
        self.feature_names = feature_names
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Feature scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        self.scalers['main'] = scaler
        
        # Feature selection
        selector = SelectKBest(score_func=f_classif, k=min(20, len(feature_names)))
        X_train_selected = selector.fit_transform(X_train_scaled, y_train)
        X_test_selected = selector.transform(X_test_scaled)
        self.feature_selectors['main'] = selector
        
        # Define models
        models_to_train = {
            'random_forest': RandomForestClassifier(
                n_estimators=100, random_state=42, class_weight='balanced'
            ),
            'gradient_boosting': GradientBoostingClassifier(
                n_estimators=100, random_state=42
            ),
            'xgboost': xgb.XGBClassifier(
                n_estimators=100, random_state=42, eval_metric='logloss'
            ),
            'logistic_regression': LogisticRegression(
                random_state=42, class_weight='balanced', max_iter=1000
            )
        }
        
        results = {}
        
        for name, model in models_to_train.items():
            print(f"  Training {name}...")
            
            # Train model
            model.fit(X_train_selected, y_train)
            
            # Predictions
            y_pred = model.predict(X_test_selected)
            y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
            
            # Evaluation metrics
            auc_score = roc_auc_score(y_test, y_pred_proba)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train_selected, y_train, cv=5, scoring='roc_auc')
            
            results[name] = {
                'auc_score': auc_score,
                'cv_mean': cv_scores.mean(),
                'cv_std': cv_scores.std(),
                'classification_report': classification_report(y_test, y_pred, output_dict=True)
            }
            
            # Store model
            self.models[name] = model
            
            print(f"    AUC: {auc_score:.3f}, CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
        
        self.is_trained = True
        
        # Find best model
        best_model_name = max(results.keys(), key=lambda k: results[k]['auc_score'])
        results['best_model'] = best_model_name
        
        print(f"✅ Best model: {best_model_name} (AUC: {results[best_model_name]['auc_score']:.3f})")
        
        return results
    
    def predict_churn(self, df: pd.DataFrame, model_name: str = None) -> List[Prediction]:
        """Predict churn probability for customers."""
        if not self.is_trained:
            raise ValueError("Models must be trained before making predictions")
        
        if model_name is None:
            model_name = 'random_forest'  # Default model
        
        X, _, _ = self.prepare_data(df)
        X_scaled = self.scalers['main'].transform(X)
        X_selected = self.feature_selectors['main'].transform(X_scaled)
        
        model = self.models[model_name]
        probabilities = model.predict_proba(X_selected)[:, 1]
        
        # Get feature importance
        if hasattr(model, 'feature_importances_'):
            selected_features = self.feature_selectors['main'].get_support()
            selected_feature_names = [self.feature_names[i] for i in range(len(selected_features)) if selected_features[i]]
            feature_importance = dict(zip(selected_feature_names, model.feature_importances_))
        else:
            feature_importance = {}
        
        predictions = []
        for i, (_, row) in enumerate(df.iterrows()):
            prediction = Prediction(
                customer_id=row['customer_id'],
                prediction_type='churn',
                probability=float(probabilities[i]),
                confidence=0.8,  # Simplified confidence calculation
                model_used=model_name,
                feature_importance=feature_importance
            )
            predictions.append(prediction)
        
        return predictions
    
    def get_feature_importance(self, model_name: str = 'random_forest', top_k: int = 10) -> Dict[str, float]:
        """Get top feature importances from a model."""
        if model_name not in self.models:
            return {}
        
        model = self.models[model_name]
        if not hasattr(model, 'feature_importances_'):
            return {}
        
        selected_features = self.feature_selectors['main'].get_support()
        selected_feature_names = [self.feature_names[i] for i in range(len(selected_features)) if selected_features[i]]
        
        feature_importance = dict(zip(selected_feature_names, model.feature_importances_))
        
        # Sort by importance and return top k
        sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)
        return dict(sorted_features[:top_k])

# Train churn prediction models
churn_model = ChurnPredictionModel()
training_results = churn_model.train_models(feature_df)

print("\n📊 MODEL PERFORMANCE SUMMARY:")
for model_name, results in training_results.items():
    if model_name != 'best_model':
        print(f"  • {model_name}: AUC {results['auc_score']:.3f} | CV {results['cv_mean']:.3f} ± {results['cv_std']:.3f}")

print(f"\n🏆 Best Model: {training_results['best_model']}")

## Customer Health Scoring System

In [ ]:
class CustomerHealthScorer:
    """Comprehensive customer health scoring system."""
    
    def __init__(self, churn_model: ChurnPredictionModel):
        self.churn_model = churn_model
        
        # Health scoring weights
        self.health_weights = {
            'usage_score': 0.30,
            'engagement_score': 0.25,
            'financial_score': 0.20,
            'support_score': 0.15,
            'churn_score': 0.10
        }
    
    def calculate_health_score(self, df: pd.DataFrame) -> List[CustomerHealth]:
        """Calculate comprehensive health scores for customers."""
        print("❤️ Calculating customer health scores...")
        
        health_scores = []
        
        # Get churn predictions
        churn_predictions = self.churn_model.predict_churn(df)
        churn_dict = {p.customer_id: p.probability for p in churn_predictions}
        
        for _, row in df.iterrows():
            customer_id = row['customer_id']
            
            # Calculate component scores
            usage_score = self._calculate_usage_score(row)
            engagement_score = self._calculate_engagement_score(row)
            financial_score = self._calculate_financial_score(row)
            support_score = self._calculate_support_score(row)
            churn_score = 100 * (1 - churn_dict.get(customer_id, 0.5))  # Invert churn probability
            
            # Calculate overall health score
            overall_score = (
                usage_score * self.health_weights['usage_score'] +
                engagement_score * self.health_weights['engagement_score'] +
                financial_score * self.health_weights['financial_score'] +
                support_score * self.health_weights['support_score'] +
                churn_score * self.health_weights['churn_score']
            )
            
            # Determine health status
            health_status = self._get_health_status(overall_score)
            risk_level = self._get_risk_level(overall_score, churn_dict.get(customer_id, 0.5))
            
            # Identify risk factors and opportunities
            risk_factors = self._identify_risk_factors(row, {
                'usage': usage_score,
                'engagement': engagement_score,
                'financial': financial_score,
                'support': support_score
            })
            
            opportunities = self._identify_opportunities(row, {
                'usage': usage_score,
                'engagement': engagement_score,
                'financial': financial_score,
                'support': support_score
            })
            
            recommended_actions = self._generate_recommendations(risk_factors, opportunities, health_status)
            
            # Calculate expansion probability
            expansion_probability = self._calculate_expansion_probability(row, overall_score)
            
            health = CustomerHealth(
                customer_id=customer_id,
                health_score=round(overall_score, 1),
                health_status=health_status,
                churn_probability=churn_dict.get(customer_id, 0.5),
                risk_level=risk_level,
                expansion_probability=expansion_probability,
                key_risk_factors=risk_factors,
                opportunities=opportunities,
                recommended_actions=recommended_actions
            )
            
            health_scores.append(health)
        
        return health_scores
    
    def _calculate_usage_score(self, row: pd.Series) -> float:
        """Calculate usage-based health score."""
        # Normalize metrics to 0-100 scale
        scores = []
        
        # Daily users score
        if row['avg_daily_users'] > 0:
            scores.append(min(100, row['avg_daily_users'] * 2))  # Adjust scaling as needed
        
        # Feature adoption score
        scores.append(row['avg_feature_adoption'])
        
        # Session quality score
        if row['avg_session_duration'] > 0:
            scores.append(min(100, row['avg_session_duration'] * 3))
        
        # API usage score
        if row['total_api_calls'] > 0:
            scores.append(min(100, row['total_api_calls'] / 100))
        
        # Usage trend score
        trend_score = 50 + (row['usage_trend'] * 10)  # Normalize around 50
        scores.append(max(0, min(100, trend_score)))
        
        return np.mean(scores) if scores else 0
    
    def _calculate_engagement_score(self, row: pd.Series) -> float:
        """Calculate engagement-based health score."""
        scores = []
        
        # Email engagement
        email_score = row['email_engagement_rate'] * 100
        scores.append(min(100, email_score))
        
        # Training completion
        training_score = row['total_training_completed'] * 25  # Each training worth 25 points
        scores.append(min(100, training_score))
        
        # Documentation engagement
        doc_score = min(100, row['total_doc_views'] * 10)
        scores.append(doc_score)
        
        # Community participation
        community_score = row['total_community_posts'] * 50
        scores.append(min(100, community_score))
        
        # CSM interaction score
        csm_score = row['total_csm_interactions'] * 30
        scores.append(min(100, csm_score))
        
        # Last login recency score
        login_score = max(0, 100 - (row['avg_last_login_days'] * 5))
        scores.append(login_score)
        
        return np.mean(scores) if scores else 0
    
    def _calculate_financial_score(self, row: pd.Series) -> float:
        """Calculate financial health score."""
        scores = []
        
        # Contract value score (tier-based)
        if row['mrr'] > 1000:
            scores.append(90)
        elif row['mrr'] > 500:
            scores.append(70)
        elif row['mrr'] > 100:
            scores.append(50)
        else:
            scores.append(30)
        
        # Contract term score
        days_remaining = row['days_to_contract_end']
        if days_remaining > 180:
            scores.append(90)
        elif days_remaining > 90:
            scores.append(70)
        elif days_remaining > 30:
            scores.append(40)
        else:
            scores.append(10)
        
        # Value per user
        if row['value_per_user'] > 50:
            scores.append(80)
        elif row['value_per_user'] > 20:
            scores.append(60)
        else:
            scores.append(40)
        
        return np.mean(scores) if scores else 0
    
    def _calculate_support_score(self, row: pd.Series) -> float:
        """Calculate support-based health score."""
        # Inverse relationship - fewer tickets = higher score
        ticket_rate = row['support_ticket_rate']
        
        if ticket_rate == 0:
            return 100
        elif ticket_rate < 0.1:
            return 80
        elif ticket_rate < 0.5:
            return 60
        elif ticket_rate < 1.0:
            return 40
        else:
            return 20
    
    def _get_health_status(self, score: float) -> HealthStatus:
        """Determine health status from score."""
        if score >= 80:
            return HealthStatus.HEALTHY
        elif score >= 60:
            return HealthStatus.AT_RISK
        elif score >= 40:
            return HealthStatus.UNHEALTHY
        else:
            return HealthStatus.CRITICAL
    
    def _get_risk_level(self, health_score: float, churn_probability: float) -> RiskLevel:
        """Determine risk level from health score and churn probability."""
        if churn_probability > 0.7 or health_score < 40:
            return RiskLevel.CRITICAL
        elif churn_probability > 0.5 or health_score < 60:
            return RiskLevel.HIGH
        elif churn_probability > 0.3 or health_score < 80:
            return RiskLevel.MEDIUM
        else:
            return RiskLevel.LOW
    
    def _identify_risk_factors(self, row: pd.Series, component_scores: Dict[str, float]) -> List[str]:
        """Identify key risk factors for the customer."""
        risk_factors = []
        
        if component_scores['usage'] < 50:
            risk_factors.append("Low product usage")
        
        if component_scores['engagement'] < 40:
            risk_factors.append("Poor engagement with support resources")
        
        if row['days_to_contract_end'] < 60:
            risk_factors.append("Contract renewal approaching")
        
        if row['avg_last_login_days'] > 14:
            risk_factors.append("Infrequent login activity")
        
        if row['support_ticket_rate'] > 0.5:
            risk_factors.append("High support ticket volume")
        
        if row['usage_trend'] < -0.1:
            risk_factors.append("Declining usage trend")
        
        if row['avg_feature_adoption'] < 30:
            risk_factors.append("Low feature adoption")
        
        return risk_factors[:5]  # Return top 5 risk factors
    
    def _identify_opportunities(self, row: pd.Series, component_scores: Dict[str, float]) -> List[str]:
        """Identify expansion and improvement opportunities."""
        opportunities = []
        
        if row['avg_daily_users'] > row['avg_monthly_users'] * 0.8:
            opportunities.append("High user engagement - potential for upsell")
        
        if row['mrr'] < 1000 and component_scores['usage'] > 70:
            opportunities.append("Heavy usage on low tier - upgrade opportunity")
        
        if row['total_training_completed'] > 2:
            opportunities.append("High training engagement - advanced features introduction")
        
        if row['avg_feature_adoption'] > 70:
            opportunities.append("High feature adoption - ready for advanced features")
        
        if row['company_size'] > 100 and row['avg_monthly_users'] < 50:
            opportunities.append("Large company with low adoption - rollout expansion")
        
        if component_scores['engagement'] > 80:
            opportunities.append("Highly engaged - potential advocate/reference customer")
        
        return opportunities[:4]  # Return top 4 opportunities
    
    def _generate_recommendations(self, risk_factors: List[str], opportunities: List[str], 
                                health_status: HealthStatus) -> List[str]:
        """Generate actionable recommendations."""
        recommendations = []
        
        # Risk-based recommendations
        if "Low product usage" in risk_factors:
            recommendations.append("Schedule usage review and training session")
        
        if "Contract renewal approaching" in risk_factors:
            recommendations.append("Initiate renewal discussion and value demonstration")
        
        if "High support ticket volume" in risk_factors:
            recommendations.append("Proactive support outreach and issue resolution")
        
        if "Declining usage trend" in risk_factors:
            recommendations.append("Investigate usage decline and provide re-engagement support")
        
        # Opportunity-based recommendations
        if "upgrade opportunity" in ' '.join(opportunities).lower():
            recommendations.append("Present tier upgrade options and ROI analysis")
        
        if "advanced features" in ' '.join(opportunities).lower():
            recommendations.append("Introduce advanced features through guided demo")
        
        if "advocate" in ' '.join(opportunities).lower():
            recommendations.append("Invite to customer advisory board or case study participation")
        
        # Health status-based recommendations
        if health_status == HealthStatus.CRITICAL:
            recommendations.append("Immediate executive intervention and recovery plan")
        elif health_status == HealthStatus.UNHEALTHY:
            recommendations.append("Schedule health check meeting with CSM")
        elif health_status == HealthStatus.AT_RISK:
            recommendations.append("Implement proactive monitoring and check-ins")
        
        return recommendations[:5]  # Return top 5 recommendations
    
    def _calculate_expansion_probability(self, row: pd.Series, health_score: float) -> float:
        """Calculate probability of expansion opportunity."""
        expansion_factors = []
        
        # High usage relative to tier
        if row['avg_daily_users'] > 50 and row['mrr'] < 1000:
            expansion_factors.append(0.3)
        
        # High feature adoption
        if row['avg_feature_adoption'] > 70:
            expansion_factors.append(0.2)
        
        # Company size vs current usage
        usage_potential = row['company_size'] / max(1, row['avg_monthly_users'])
        if usage_potential > 5:
            expansion_factors.append(0.25)
        
        # High engagement
        if row['total_training_completed'] > 1 or row['total_csm_interactions'] > 2:
            expansion_factors.append(0.15)
        
        # Health score bonus
        if health_score > 80:
            expansion_factors.append(0.1)
        
        return min(0.9, sum(expansion_factors))

# Calculate health scores
health_scorer = CustomerHealthScorer(churn_model)
health_scores = health_scorer.calculate_health_score(feature_df)

print(f"✅ Calculated health scores for {len(health_scores)} customers")

# Health distribution
health_distribution = {}
for health in health_scores:
    status = health.health_status.value
    health_distribution[status] = health_distribution.get(status, 0) + 1

print("\n❤️ HEALTH DISTRIBUTION:")
for status, count in health_distribution.items():
    percentage = (count / len(health_scores)) * 100
    print(f"  • {status.title()}: {count} ({percentage:.1f}%)")

# Risk distribution
risk_distribution = {}
for health in health_scores:
    risk = health.risk_level.value
    risk_distribution[risk] = risk_distribution.get(risk, 0) + 1

print("\n⚠️ RISK DISTRIBUTION:")
for risk, count in risk_distribution.items():
    percentage = (count / len(health_scores)) * 100
    print(f"  • {risk.title()} Risk: {count} ({percentage:.1f}%)")

## AI-Powered Insights and Recommendations

In [ ]:
class AIInsightsEngine:
    """AI-powered insights and recommendations engine."""
    
    def __init__(self):
        self.openai_client = openai_client
        self.anthropic_client = anthropic_client if ANTHROPIC_AVAILABLE else None
    
    async def generate_customer_insights(self, customer: Customer, health: CustomerHealth, 
                                       recent_data: Dict[str, Any]) -> Dict[str, Any]:
        """Generate AI-powered insights for a specific customer."""
        try:
            insights_prompt = f"""
            Analyze this customer's data and provide strategic insights:
            
            Customer Profile:
            - Company: {customer.company}
            - Tier: {customer.tier.value}
            - MRR: ${customer.mrr:,.0f}
            - Industry: {customer.industry}
            - Company Size: {customer.company_size}
            - Contract End: {customer.contract_end_date.strftime('%Y-%m-%d')}
            - Days as Customer: {(datetime.now() - customer.signup_date).days}
            
            Health Metrics:
            - Health Score: {health.health_score}/100
            - Health Status: {health.health_status.value}
            - Churn Probability: {health.churn_probability:.1%}
            - Risk Level: {health.risk_level.value}
            - Expansion Probability: {health.expansion_probability:.1%}
            
            Key Risk Factors:
            {chr(10).join(f'- {factor}' for factor in health.key_risk_factors)}
            
            Opportunities:
            {chr(10).join(f'- {opp}' for opp in health.opportunities)}
            
            Usage Data:
            - Daily Active Users: {recent_data.get('avg_daily_users', 0):.0f}
            - Feature Adoption: {recent_data.get('avg_feature_adoption', 0):.1f}%
            - Session Duration: {recent_data.get('avg_session_duration', 0):.1f} min
            - Support Tickets: {recent_data.get('total_support_tickets', 0)}
            
            Provide a comprehensive analysis including:
            1. Executive Summary (2-3 sentences)
            2. Key Insights (3-4 bullet points)
            3. Strategic Recommendations (3-4 specific actions)
            4. Risk Assessment (specific concerns and mitigation strategies)
            5. Growth Opportunities (expansion and upsell potential)
            
            Format as JSON with keys: executive_summary, key_insights, strategic_recommendations, risk_assessment, growth_opportunities
            """
            
            response = self.openai_client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You are a customer success expert providing strategic insights for B2B SaaS companies. Provide actionable, data-driven recommendations."},
                    {"role": "user", "content": insights_prompt}
                ],
                temperature=0.3
            )
            
            # Parse JSON response
            insights_text = response.choices[0].message.content
            
            # Extract JSON from response
            import re
            json_match = re.search(r'\{.*\}', insights_text, re.DOTALL)
            if json_match:
                insights = json.loads(json_match.group())
            else:
                # Fallback structured response
                insights = {
                    "executive_summary": f"Customer shows {health.health_status.value} health with {health.churn_probability:.1%} churn risk.",
                    "key_insights": health.key_risk_factors[:3],
                    "strategic_recommendations": health.recommended_actions[:3],
                    "risk_assessment": f"{health.risk_level.value} risk level requires attention",
                    "growth_opportunities": health.opportunities[:2]
                }
            
            return {
                'status': 'success',
                'insights': insights,
                'generated_at': datetime.now().isoformat()
            }
            
        except Exception as e:
            return {
                'status': 'error',
                'error': str(e),
                'fallback_insights': {
                    'executive_summary': f"Customer health score: {health.health_score}/100",
                    'key_insights': health.key_risk_factors,
                    'strategic_recommendations': health.recommended_actions,
                    'risk_assessment': f"Risk level: {health.risk_level.value}",
                    'growth_opportunities': health.opportunities
                }
            }
    
    def generate_portfolio_insights(self, health_scores: List[CustomerHealth], 
                                  feature_df: pd.DataFrame) -> Dict[str, Any]:
        """Generate insights for the entire customer portfolio."""
        try:
            # Calculate portfolio metrics
            total_customers = len(health_scores)
            avg_health_score = np.mean([h.health_score for h in health_scores])
            avg_churn_risk = np.mean([h.churn_probability for h in health_scores])
            
            # Risk distribution
            risk_counts = {}
            for health in health_scores:
                risk = health.risk_level.value
                risk_counts[risk] = risk_counts.get(risk, 0) + 1
            
            # Health distribution
            health_counts = {}
            for health in health_scores:
                status = health.health_status.value
                health_counts[status] = health_counts.get(status, 0) + 1
            
            # Revenue at risk
            high_risk_customers = [h for h in health_scores if h.risk_level in [RiskLevel.HIGH, RiskLevel.CRITICAL]]
            revenue_at_risk = sum([feature_df[feature_df['customer_id'] == h.customer_id]['mrr'].iloc[0] 
                                 for h in high_risk_customers if len(feature_df[feature_df['customer_id'] == h.customer_id]) > 0])
            
            # Expansion opportunities
            expansion_customers = [h for h in health_scores if h.expansion_probability > 0.5]
            expansion_revenue_potential = sum([feature_df[feature_df['customer_id'] == h.customer_id]['mrr'].iloc[0] * 0.3
                                             for h in expansion_customers if len(feature_df[feature_df['customer_id'] == h.customer_id]) > 0])
            
            portfolio_insights = {
                'overview': {
                    'total_customers': total_customers,
                    'average_health_score': round(avg_health_score, 1),
                    'average_churn_risk': f"{avg_churn_risk:.1%}",
                    'revenue_at_risk': f"${revenue_at_risk:,.0f}/month",
                    'expansion_potential': f"${expansion_revenue_potential:,.0f}/month"
                },
                'risk_distribution': risk_counts,
                'health_distribution': health_counts,
                'key_findings': [
                    f"{risk_counts.get('critical', 0) + risk_counts.get('high', 0)} customers at high/critical risk",
                    f"{len(expansion_customers)} customers with expansion opportunities",
                    f"{health_counts.get('healthy', 0)} customers in healthy status",
                    f"${revenue_at_risk:,.0f} monthly revenue at risk from churn"
                ],
                'recommendations': [
                    "Focus immediate attention on critical risk customers",
                    "Develop expansion campaigns for high-potential accounts",
                    "Implement proactive health monitoring for at-risk customers",
                    "Create customer success playbooks based on risk factors"
                ]
            }
            
            return {
                'status': 'success',
                'insights': portfolio_insights,
                'generated_at': datetime.now().isoformat()
            }
            
        except Exception as e:
            return {
                'status': 'error',
                'error': str(e)
            }
    
    def generate_intervention_plan(self, customer_id: str, health: CustomerHealth, 
                                 urgency: str = "normal") -> Dict[str, Any]:
        """Generate a specific intervention plan for a customer."""
        try:
            if health.risk_level == RiskLevel.CRITICAL:
                urgency = "immediate"
            elif health.risk_level == RiskLevel.HIGH:
                urgency = "urgent"
            
            intervention_plan = {
                'customer_id': customer_id,
                'urgency_level': urgency,
                'timeline': self._get_intervention_timeline(health.risk_level),
                'primary_objectives': self._get_primary_objectives(health),
                'action_items': self._get_action_items(health),
                'success_metrics': self._get_success_metrics(health),
                'escalation_triggers': self._get_escalation_triggers(health),
                'resources_needed': self._get_required_resources(health)
            }
            
            return {
                'status': 'success',
                'intervention_plan': intervention_plan,
                'created_at': datetime.now().isoformat()
            }
            
        except Exception as e:
            return {
                'status': 'error',
                'error': str(e)
            }
    
    def _get_intervention_timeline(self, risk_level: RiskLevel) -> str:
        timelines = {
            RiskLevel.CRITICAL: "Immediate action required (24-48 hours)",
            RiskLevel.HIGH: "Urgent action required (1 week)",
            RiskLevel.MEDIUM: "Action required (2-4 weeks)",
            RiskLevel.LOW: "Monitor and maintain (monthly check-ins)"
        }
        return timelines.get(risk_level, "Standard timeline (2 weeks)")
    
    def _get_primary_objectives(self, health: CustomerHealth) -> List[str]:
        objectives = []
        
        if health.churn_probability > 0.7:
            objectives.append("Prevent imminent churn")
        elif health.churn_probability > 0.5:
            objectives.append("Reduce churn risk")
        
        if "Low product usage" in health.key_risk_factors:
            objectives.append("Increase product adoption")
        
        if health.expansion_probability > 0.5:
            objectives.append("Explore expansion opportunities")
        
        if health.health_score < 60:
            objectives.append("Improve overall customer health")
        
        return objectives[:3]  # Top 3 objectives
    
    def _get_action_items(self, health: CustomerHealth) -> List[Dict[str, str]]:
        actions = []
        
        for recommendation in health.recommended_actions:
            action = {
                'task': recommendation,
                'owner': 'CSM',
                'priority': 'high' if health.risk_level in [RiskLevel.CRITICAL, RiskLevel.HIGH] else 'medium',
                'due_date': (datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d')
            }
            actions.append(action)
        
        return actions
    
    def _get_success_metrics(self, health: CustomerHealth) -> List[str]:
        metrics = [
            f"Improve health score to {min(100, health.health_score + 20)}",
            f"Reduce churn probability to {max(0.1, health.churn_probability - 0.2):.1%}",
            "Increase user engagement by 25%",
            "Achieve 90% response rate to outreach"
        ]
        
        return metrics[:3]
    
    def _get_escalation_triggers(self, health: CustomerHealth) -> List[str]:
        triggers = [
            "No response to outreach within 48 hours",
            "Further decline in usage metrics",
            "Additional support tickets raised",
            "Contract renewal rejection"
        ]
        
        return triggers
    
    def _get_required_resources(self, health: CustomerHealth) -> List[str]:
        resources = ["Customer Success Manager time"]
        
        if health.risk_level == RiskLevel.CRITICAL:
            resources.extend(["Executive involvement", "Technical support specialist"])
        
        if "training" in ' '.join(health.recommended_actions).lower():
            resources.append("Training resources and materials")
        
        if health.expansion_probability > 0.5:
            resources.append("Sales engineer for upsell discussion")
        
        return resources

# Initialize AI insights engine
ai_insights = AIInsightsEngine()
print("✅ AI Insights Engine ready!")

## Demo: Customer Success Prediction System

In [ ]:
# Demo the complete customer success prediction system
print("🚀 CUSTOMER SUCCESS PREDICTION SYSTEM DEMO")
print("=" * 50)

# Select sample customers for detailed analysis
sample_customers = [
    health_scores[0],   # First customer
    health_scores[50],  # Middle customer
    health_scores[100], # Another customer
]

# Find high-risk and high-opportunity customers
high_risk_customers = [h for h in health_scores if h.risk_level in [RiskLevel.CRITICAL, RiskLevel.HIGH]][:3]
high_opportunity_customers = [h for h in health_scores if h.expansion_probability > 0.6][:3]

print(f"\n🎯 SAMPLE CUSTOMER ANALYSIS:")
print("-" * 30)

for i, health in enumerate(sample_customers, 1):
    customer = next((c for c in customers if c.id == health.customer_id), None)
    if customer:
        customer_data = feature_df[feature_df['customer_id'] == health.customer_id].iloc[0]
        
        print(f"\n{i}. Customer {health.customer_id} ({customer.company})")
        print(f"   💰 MRR: ${customer.mrr:,.0f} | Tier: {customer.tier.value}")
        print(f"   ❤️  Health Score: {health.health_score}/100 ({health.health_status.value})")
        print(f"   ⚠️  Churn Risk: {health.churn_probability:.1%} ({health.risk_level.value} risk)")
        print(f"   📈 Expansion Probability: {health.expansion_probability:.1%}")
        
        print(f"   🔴 Risk Factors:")
        for risk in health.key_risk_factors[:3]:
            print(f"     - {risk}")
        
        print(f"   🟢 Opportunities:")
        for opp in health.opportunities[:2]:
            print(f"     - {opp}")
        
        print(f"   💡 Top Recommendations:")
        for rec in health.recommended_actions[:2]:
            print(f"     - {rec}")

print(f"\n⚠️ HIGH-RISK CUSTOMERS REQUIRING IMMEDIATE ATTENTION:")
print("-" * 50)

for i, health in enumerate(high_risk_customers, 1):
    customer = next((c for c in customers if c.id == health.customer_id), None)
    if customer:
        print(f"\n{i}. {customer.company} (Customer {health.customer_id})")
        print(f"   💰 Monthly Revenue at Risk: ${customer.mrr:,.0f}")
        print(f"   📊 Health Score: {health.health_score}/100")
        print(f"   🚨 Churn Probability: {health.churn_probability:.1%}")
        print(f"   ⏰ Risk Level: {health.risk_level.value.upper()}")
        
        # Generate intervention plan
        intervention = ai_insights.generate_intervention_plan(health.customer_id, health)
        if intervention['status'] == 'success':
            plan = intervention['intervention_plan']
            print(f"   📋 Intervention Timeline: {plan['timeline']}")
            print(f"   🎯 Primary Objectives: {', '.join(plan['primary_objectives'][:2])}")
            
            if plan['action_items']:
                print(f"   ✅ Next Actions:")
                for action in plan['action_items'][:2]:
                    print(f"     - {action['task']} (Due: {action['due_date']})")

print(f"\n📈 HIGH-OPPORTUNITY CUSTOMERS FOR EXPANSION:")
print("-" * 45)

total_expansion_potential = 0
for i, health in enumerate(high_opportunity_customers, 1):
    customer = next((c for c in customers if c.id == health.customer_id), None)
    if customer:
        expansion_value = customer.mrr * 0.3  # Assume 30% expansion potential
        total_expansion_potential += expansion_value
        
        print(f"\n{i}. {customer.company} (Customer {health.customer_id})")
        print(f"   💰 Current MRR: ${customer.mrr:,.0f}")
        print(f"   📊 Health Score: {health.health_score}/100")
        print(f"   🚀 Expansion Probability: {health.expansion_probability:.1%}")
        print(f"   💎 Potential Additional Revenue: ${expansion_value:,.0f}/month")
        
        print(f"   🎯 Expansion Opportunities:")
        for opp in health.opportunities[:2]:
            print(f"     - {opp}")

print(f"\n💰 Total Expansion Potential: ${total_expansion_potential:,.0f}/month")

# Portfolio-level insights
print(f"\n📊 PORTFOLIO-LEVEL INSIGHTS:")
print("-" * 28)

portfolio_insights = ai_insights.generate_portfolio_insights(health_scores, feature_df)
if portfolio_insights['status'] == 'success':
    insights = portfolio_insights['insights']
    overview = insights['overview']
    
    print(f"\n📈 Overview:")
    print(f"  • Total Customers: {overview['total_customers']}")
    print(f"  • Average Health Score: {overview['average_health_score']}/100")
    print(f"  • Average Churn Risk: {overview['average_churn_risk']}")
    print(f"  • Revenue at Risk: {overview['revenue_at_risk']}")
    print(f"  • Expansion Potential: {overview['expansion_potential']}")
    
    print(f"\n🎯 Key Findings:")
    for finding in insights['key_findings']:
        print(f"  • {finding}")
    
    print(f"\n💡 Strategic Recommendations:")
    for rec in insights['recommendations']:
        print(f"  • {rec}")

# Model performance summary
print(f"\n🤖 ML MODEL PERFORMANCE:")
print("-" * 25)

feature_importance = churn_model.get_feature_importance('random_forest', top_k=10)
print(f"\n🔝 Top Predictive Features:")
for i, (feature, importance) in enumerate(feature_importance.items(), 1):
    print(f"  {i}. {feature}: {importance:.3f}")

print(f"\n📊 Model Accuracy:")
best_model = training_results['best_model']
best_results = training_results[best_model]
print(f"  • Best Model: {best_model}")
print(f"  • AUC Score: {best_results['auc_score']:.3f}")
print(f"  • Cross-Validation: {best_results['cv_mean']:.3f} ± {best_results['cv_std']:.3f}")

print("\n" + "=" * 50)
print("🎉 Customer Success Prediction Demo Complete!")

## Summary

The **Customer Success Prediction Agent** demonstrates a comprehensive approach to customer success management using machine learning and AI. The system provides:

### Key Features:
- **ML-Powered Churn Prediction**: Multiple models (Random Forest, XGBoost, Gradient Boosting) for accurate churn prediction
- **Comprehensive Health Scoring**: Multi-dimensional health assessment combining usage, engagement, financial, and support metrics
- **AI-Generated Insights**: GPT-4 powered strategic recommendations and intervention plans
- **Risk Stratification**: Automated risk level assignment with actionable recommendations
- **Expansion Identification**: Proactive identification of upsell and cross-sell opportunities
- **Portfolio Analytics**: Company-wide customer success metrics and trends

### Technical Architecture:
- **Feature Engineering**: 25+ engineered features from customer, usage, and engagement data
- **Model Ensemble**: Multiple ML algorithms with cross-validation and feature selection
- **Real-time Scoring**: Dynamic health score calculation with trend analysis
- **AI Integration**: OpenAI GPT-4 for strategic insights and recommendations
- **Intervention Planning**: Automated action plan generation based on risk levels

### Business Applications:
- Proactive churn prevention and customer retention
- Revenue expansion through targeted upsell campaigns
- Customer success team optimization and prioritization
- Executive reporting and strategic decision making
- Automated customer success workflows

### Production Capabilities:
- Model retraining and performance monitoring
- Real-time data pipeline integration
- CRM and customer data platform connectivity
- Automated alerting and intervention triggers
- ROI tracking and success measurement

This agent showcases how machine learning and AI can be combined to create a sophisticated customer success platform that proactively identifies risks and opportunities while providing actionable insights for customer success teams.